# OECD SDBS: integrazione nel confronto internazionale

            Questo foglio mostra come la fonte OECD entra nel progetto. La pipeline attuale produce un inventario operativo; il notebook lo collega al perimetro Eurostat e alla mappa settoriale gia definita.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from valore_aggiunto_imprese.analysis import (
    METRIC_LABELS,
    add_source_note,
    latest_common_year_with_values,
    latest_year_with_values,
    SIZE_ORDER_BUSINESS,
    SIZE_ORDER_OFFICIAL,
    add_share,
    enrich_business_demography,
    enrich_sbs,
    read_project_csv,
)

pd.options.display.max_columns = 80
pd.options.display.float_format = "{:,.2f}".format

## Inventario disponibile

In [ ]:
oecd_inventory = read_project_csv("data/processed/oecd_sdbs_inventory.csv")
oecd_inventory

## Ponte settoriale NACE-ISIC

            Il confronto OECD richiede un ponte tra NACE ed ISIC. La tabella mostra i settori del progetto che hanno gia un codice ISIC previsto.

In [ ]:
from valore_aggiunto_imprese.config import SECTORS

sector_bridge = pd.DataFrame(SECTORS).rename(
    columns={
        "sector_code_harmonised": "settore_progetto",
        "label_it": "settore",
        "eurostat_nace": "codice_nace_eurostat",
        "oecd_isic": "codice_isic_oecd",
    }
)
sector_bridge["stato"] = sector_bridge["codice_isic_oecd"].notna().map(
    {True: "Con codice ISIC", False: "Da mappare"}
)
sector_bridge[
    ["settore_progetto", "settore", "codice_nace_eurostat", "codice_isic_oecd", "stato"]
]

## Copertura della mappa settoriale

In [ ]:
bridge_summary = sector_bridge.groupby("stato", as_index=False).size()

fig = px.bar(
    bridge_summary,
    x="stato",
    y="size",
    color="stato",
    labels={"stato": "Stato", "size": "Numero settori"},
    title="Settori del progetto predisposti per OECD SDBS",
)
add_source_note(fig, "OECD SDBS; configurazione del progetto")
fig

## Output prodotti

In [ ]:
output_files = pd.DataFrame(
    [
        {"file": "data/processed/oecd_sdbs_inventory.csv", "ruolo": "inventario fonte"},
        {"file": "data/processed_csv/oecd_sdbs_inventory.csv", "ruolo": "export CSV"},
        {"file": "data/processed_json/oecd_sdbs_inventory.json", "ruolo": "export JSON"},
    ]
)
output_files["presente"] = output_files["file"].map(lambda path: (PROJECT_ROOT / path).exists())
output_files